In [ ]:
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

with open('manifestacoes.json', 'r', encoding='utf-8') as f:
    dados = json.load(f)

textos = [d['texto'] for d in dados]
ids = [d['id'] for d in dados]

modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = modelo.encode(textos)

def detectar_duplicatas(textos, emb, lista_ids, limiar=0.85):
    matriz_sim = cosine_similarity(emb)
    duplicatas = []
    n = len(textos)
    
    for i in range(n):
        for j in range(i+1, n):
            if matriz_sim[i][j] >= limiar:
                duplicatas.append({
                    "ID 1": lista_ids[i],
                    "ID 2": lista_ids[j],
                    "Score": matriz_sim[i][j]
                })
    return matriz_sim, duplicatas

matriz, pares = detectar_duplicatas(textos, embeddings, ids, limiar=0.85)

# Matriz de Similaridade Completa (Heatmap)
plt.figure(figsize=(10, 8))
sns.heatmap(matriz, cmap="magma", xticklabels=False, yticklabels=False)
plt.title("Heatmap de Similaridade Semântica")
plt.show()

print(f"Pares Duplicados Encontrados (Limiar 0.85): {len(pares)}")
display(pd.DataFrame(pares))

# Justificativa do Limiar e Análise (para o Notebook):
# O limiar de 0.85 garante que falsos positivos (como manifestações distintas dentro de uma mesma categoria genérica, 
# ex: duas ruas diferentes com buracos) não sejam agrupados, enquanto mitiga falsos negativos ao capturar as duplicatas 
# reais com variação vocabular.